In [ ]:
import pandas as pd
import altair as alt
import json
from IPython.display import display, HTML

# 1. Cargar el archivo movimientos_y_tabla.csv y limpiar datos con Pandas
df = pd.read_csv('movimientos_y_tabla.csv', sep=';', encoding='latin-1')

# Diccionario de corrección exacta para los nombres mal codificados en el CSV
mapeo_equipos = {
    'CD Universidad Cat\x97lica': 'CD Universidad Católica',
    'CD Universidad Cat—lica': 'CD Universidad Católica',
    'Uni\x94n La Calera': 'Unión La Calera',
    'Uni—n La Calera': 'Unión La Calera',
    'Universidad de Concepci\x94n': 'Universidad de Concepción',
    'Universidad de Concepci—n': 'Universidad de Concepción',
    'Uni\x94n Espa\x91ola': 'Unión Española',
    'Uni—n Espa–ola': 'Unión Española',
    'CD O\'Higgins': "CD O'Higgins",
    'Deportes Copiap\x97': 'Deportes Copiapó',
    'Deportes Copiap—': 'Deportes Copiapó'
}

# Aplicar corrección: si el nombre está en el mapa se corrige, si no, se mantiene
df['Equipo'] = df['Equipo'].str.strip().replace(mapeo_equipos)

# Asegurar la conversión correcta de tipos de datos numéricos
df['Posicion'] = pd.to_numeric(df['Posicion'], errors='coerce')
df['Total altas'] = pd.to_numeric(df['Total altas'], errors='coerce')

# Filtrar estrictamente por el rango solicitado: de 2019 a 2025
df_filtrado = df[(df['Temporada'] >= 2019) & (df['Temporada'] <= 2025)].copy()
df_filtrado['Temporada'] = df_filtrado['Temporada'].astype(int)

# Crear la columna de rangos para la leyenda
df_filtrado['Rango Altas'] = df_filtrado['Total altas'].apply(
    lambda x: '20+ altas' if x >= 20 else '19- altas'
)

# 2. Configurar el menú desplegable compatible con Altair v5
opciones_temporadas = [2019, 2020, 2021, 2022, 2023, 2024, 2025]

input_dropdown = alt.binding_select(
    options=opciones_temporadas,
    name='Temporada: '
)

# SOLUCIÓN PARA ALTAIR V5: Usar selection_point con 'value' directo
selector_temporada = alt.selection_point(
    fields=['Temporada'],
    bind=input_dropdown,
    value=2025  # Valor inicial por defecto
)

# 3. Construcción del gráfico con capas vinculadas al selector
base = alt.Chart(df_filtrado).add_params(
    selector_temporada
).transform_filter(
    selector_temporada
).transform_calculate(
    etiqueta="datum.Equipo + ' (' + datum['Total altas'] + ' altas)'"
)

# Capa 1: Las barras horizontales
tabla = base.mark_bar(cornerRadiusEnd=4).encode(
    y=alt.Y('Posicion:O',
            title='Posición en la Tabla',
            sort='ascending'),
    x=alt.X('Total altas:Q',
            title='Cantidad de Fichajes (Altas)'),
    color=alt.Color('Rango Altas:N',
                    scale=alt.Scale(domain=['20+ altas', '19- altas'],
                                   range=['#00aa44', '#000000']),
                    legend=alt.Legend(
                        title="Rango de Fichajes",
                        orient="right",
                        titleFontSize=12,
                        labelFontSize=11
                    )),
    tooltip=[
        alt.Tooltip('Posicion:Q', title='Lugar en Tabla'),
        alt.Tooltip('Equipo:N', title='Club'),
        alt.Tooltip('Total altas:Q', title='Fichajes Totales'),
        alt.Tooltip('Altas por traspaso:Q', title='Por Traspaso'),
        alt.Tooltip('Altas libre:Q', title='Libres')
    ]
).properties(
    title='Tabla de Posiciones vs. Fichajes',
    width=600,
    height=500
)

# Capa 2: Las etiquetas de texto al costado de las barras
etiquetas = base.mark_text(
    align='left',
    baseline='middle',
    dx=5,
    fontWeight='bold'
).encode(
    y=alt.Y('Posicion:O', sort='ascending'),
    x=alt.X('Total altas:Q'),
    text=alt.Text('etiqueta:N'),
    color=alt.Color('Rango Altas:N',
                    scale=alt.Scale(domain=['20+ altas', '19- altas'],
                                   range=['#00aa44', '#000000']),
                    legend=None)
)

# Combinar capas y aplicar estilos globales
chart_final = (tabla + etiquetas).configure_axis(
    labelFontSize=12,
    titleFontSize=14
).configure_title(
    fontSize=18,
    anchor='start'
)

# 4. Forzar el renderizado seguro inyectando Vega de forma directa (v5 compatible)
vega_spec = chart_final.to_dict()

html_render = f"""
<div id="altair-container" style="background-color: white; padding: 10px;"></div>
<script>
  (function() {{
    function loadScript(src) {{
      return new Promise(function(resolve, reject) {{
        var s = document.createElement('script');
        s.src = src;
        s.onload = resolve;
        s.onerror = reject;
        document.head.appendChild(s);
      }});
    }}

    // Cargar las librerías CDN actualizadas a Vega-Lite v5
    loadScript("https://cdn.jsdelivr.net/npm/vega@5")
      .then(function() {{ return loadScript("https://cdn.jsdelivr.net/npm/vega-lite@5"); }})
      .then(function() {{ return loadScript("https://cdn.jsdelivr.net/npm/vega-embed@6"); }})
      .then(function() {{
        var spec = {json.dumps(vega_spec)};
        vegaEmbed('#altair-container', spec, {{"actions": false}});
      }})
      .catch(console.error);
   biographical_context_placeholder = true;
  }})();
</script>
"""

display(HTML(html_render))